# Univariate MMAR: From Simulation to Posterior Prediction

Financial returns often show two features that a constant-variance Gaussian model misses: unusually large changes occur more frequently, and periods of high volatility cluster in time. The **Multifractal Model of Asset Returns (MMAR)** represents both effects with heavy-tailed innovations evaluated on a random trading clock.

This notebook follows the complete posterior-estimation workflow for one return series:

1. define the generative model and priors;
2. check simulations from the prior;
3. specify the BayesFlow adapter and neural posterior estimator;
4. restore the included estimator, whose expensive training step has already been run;
5. check recovery and calibration on simulated data;
6. estimate the posterior for a 256-day VOO window;
7. compare posterior predictions with the observations and a Gaussian fit.

The example is adapted from [amortized-fractals](https://github.com/stefanradev93/amortized-fractals). It is a modeling demonstration rather than investment advice.

## 1. Generative Model

Let $P_t$ be the closing price on trading day $t$. The daily simple return is

$$
r_t=\frac{P_t-P_{t-1}}{P_{t-1}}.
$$

MMAR models a 256-day return window as

$$
r_t
=
\mu+\bar{\sigma}\sqrt{\theta_t(q)}\,\varepsilon_t,
\qquad
\varepsilon_t
=
z_t\sqrt{\frac{\nu-2}{U_t}},
$$

with

$$
z_t\overset{\mathrm{iid}}{\sim}\mathcal N(0,1),
\qquad
U_t\overset{\mathrm{iid}}{\sim}\chi^2_\nu,
\qquad
z_t\perp U_t.
$$

The symbols have the following meanings:

- $r_t$ is the observed daily return.
- $\mu$ is the daily drift.
- $\bar{\sigma}>0$ is the baseline root-mean-square return scale.
- $\theta_t(q)>0$ is the random trading-time intensity on day $t$. Its average over the window is one.
- $q\in(0.5,1)$ controls the contrast between quiet and turbulent periods.
- $\varepsilon_t$ is a standardized Student-$t$ innovation.
- $\nu>2$ is its degrees of freedom. Small values produce heavier tails.
- $z_t$ is a standard-normal draw and $U_t$ is an independent chi-square draw. The factor $\sqrt{(\nu-2)/U_t}$ gives $\varepsilon_t$ variance one.

Conditional on the trading clock, the innovations are independent but have daily scale $\bar{\sigma}\sqrt{\theta_t(q)}$. Marginalizing over the random clock creates dependence in return magnitudes and therefore volatility clustering.

### The Multifractal Trading Clock

The simulator begins with one interval and recursively divides it in half. At each split, one child receives relative intensity $2q$ and the other receives $2(1-q)$; their left-right orientation is random. Three levels produce eight intensities, each repeated for 32 days. The resulting 256 weights are normalized to average one and circularly shifted so that cascade boundaries do not always occur on the same dates.

![A binomial cascade creates clustered volatility across a 256-day return window.](helpers/mmar/fractal_cascade.gif)

The four unknown parameters are collected as

$$
\boldsymbol\phi=(\mu,\bar{\sigma},q,\nu).
$$

The latent cascade, its random shift, and the heavy-tailed innovations make direct likelihood calculations inconvenient. We can nevertheless simulate from $p(\mathbf r\mid\boldsymbol\phi)$, which is enough for neural posterior estimation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bayesflow as bf

from tutorials.helpers import mmar

In [ ]:
N_PRIOR_PREDICTIVE = 2_000
N_VALIDATION_CASES = 200
N_VALIDATION_SAMPLES = 400
N_POSTERIOR_SAMPLES = 2_000

mmar.configure_plot_style()

### Priors

The four parameters are drawn independently:

$$
\begin{aligned}
\mu &\sim \mathcal N(0, 0.0015^2), \\
\bar\sigma &\sim \mathcal U(0.005, 0.025), \\
q &\sim \mathcal U(0.52, 0.90), \\
\log(\nu-2) &\sim \mathcal U(\log 0.05, \log 10).
\end{aligned}
$$

The drift prior is centered near zero. The scale and cascade contrast use bounded uniform priors. The last prior guarantees $\nu>2$, so the innovation variance exists, while allocating substantial probability to heavy tails.

In [ ]:
mmar.prior_table()

## 2. The Observed VOO Window

The bundled data contain adjusted-close simple returns for VOO from January 3, 2020 through January 7, 2021. This 256-trading-day window includes the sharp COVID-era market movements and provides a useful stress test for the model.

In [ ]:
voo_returns = mmar.load_market_returns()

fig, ax = plt.subplots(figsize=(9, 3), layout="constrained")
ax.plot(voo_returns.index, 100 * voo_returns, color="#30343B", linewidth=1)
ax.axhline(0, color="#777777", linewidth=0.7)
ax.set(
    title="VOO daily simple returns",
    xlabel="Date",
    ylabel="Return (%)",
)
voo_returns.describe()

## 3. Prior Predictive Check

Before using the observations for inference, we draw parameters from the prior and simulate complete return paths. The observed marginal returns and maximum drawdown should fall within a plausible part of the prior predictive distribution. A severe mismatch here would indicate that the priors or simulator need revision before training.

In [ ]:
prior_simulations = mmar.simulate(N_PRIOR_PREDICTIVE)
fig = mmar.plot_prior_predictive(
    prior_simulations["returns"], voo_returns
)

## 4. Neural Posterior Estimation

Training data are generated by repeatedly sampling

$$
\boldsymbol\phi^{(i)}\sim p(\boldsymbol\phi),
\qquad
\mathbf r^{(i)}\sim
p(\mathbf r\mid\boldsymbol\phi^{(i)}).
$$

A summary network compresses each 256-day series into learned features. A conditional normalizing flow then represents

$$
q_\psi(\boldsymbol\phi\mid\mathbf r)
\approx
p(\boldsymbol\phi\mid\mathbf r).
$$

The network parameters $\psi$ are learned by minimizing the expected negative log probability of the simulated ground-truth parameters:

$$
\mathcal L(\psi)
=
-\mathbb E_{p(\boldsymbol\phi,\mathbf r)}
\left[
\log q_\psi(\boldsymbol\phi\mid\mathbf r)
\right].
$$

The adapter maps the three bounded parameters to unconstrained coordinates during training and converts posterior draws back to their original units. It also marks the returns as a time series for the summary network.

In [ ]:
adapter = (
    bf.Adapter()
    .convert_dtype("float64", "float32")
    .constrain(
        "sigma_bar",
        lower=mmar.PRIOR.sigma_low,
        upper=mmar.PRIOR.sigma_high,
    )
    .constrain(
        "q",
        lower=mmar.PRIOR.q_low,
        upper=mmar.PRIOR.q_high,
    )
    .constrain(
        "nu",
        lower=mmar.PRIOR.nu_low,
        upper=mmar.PRIOR.nu_high,
    )
    .concatenate(
        mmar.PARAMETER_NAMES,
        into="inference_variables",
    )
    .as_time_series("returns")
    .rename("returns", "summary_variables")
)

In [ ]:
summary_network = bf.networks.FusionTransformer(dropout=0.1)
inference_network = bf.networks.CouplingFlow(
    transform="spline",
    depth=4,
)

workflow = bf.BasicWorkflow(
    adapter=adapter,
    summary_network=summary_network,
    inference_network=inference_network,
    standardize="all",
    checkpoint_filepath=mmar.ASSET_DIR,
    checkpoint_name="univariate",
    restore=True
)

### Training Step

The included checkpoint was trained once on 100,000 simulated paths, with 500 additional simulations for validation, for 60 epochs. That expensive fit is intentionally skipped here. The training call that produced it was:

    training_simulations = mmar.simulate(100_000)
    validation_simulations = mmar.simulate(500)
    history = workflow.fit_offline(
        training_simulations,
        validation_data=validation_simulations,
        epochs=60,
        batch_size=128,
    )

After training, amortization makes inference fast: the same estimator can process any new 256-day series drawn from the modeled parameter range.

## 5. Recovery and Calibration

We now simulate fresh datasets with known parameters. Recovery compares posterior estimates with those true values. Simulation-based calibration checks whether posterior uncertainty has the advertised coverage across repeated simulated datasets. These diagnostics test the estimator on the model it was trained to invert.

In [ ]:
test_simulations = mmar.simulate(N_VALIDATION_CASES)
test_estimates = workflow.sample(
    conditions=test_simulations,
    num_samples=N_VALIDATION_SAMPLES,
)

In [ ]:
fig = bf.diagnostics.recovery(
    estimates=test_estimates,
    targets=test_simulations,
    variable_names=mmar.PARAMETER_LABELS,
    uncertainty_agg_kwargs={"prob": 0.68},
    num_row=2,
    num_col=2,
)

In [ ]:
fig = bf.diagnostics.calibration_ecdf(
    estimates=test_estimates,
    targets=test_simulations,
    variable_names=mmar.PARAMETER_LABELS,
    difference=True,
    stacked=False,
    num_row=2,
    num_col=2,
)

## 6. Posterior for VOO

The observed series is passed through the same adapter and summary network. The flow then produces draws from the learned approximation to

$$
p(\mu,\bar{\sigma},q,\nu\mid r_{1:256}^{\mathrm{VOO}}).
$$

The posterior is a distribution rather than a single fitted parameter vector, so uncertainty can be propagated into every subsequent prediction.

In [ ]:
voo_condition = {
    "returns": voo_returns.to_numpy()[None, :, None]
}
voo_samples = workflow.sample(
    conditions=voo_condition,
    num_samples=N_POSTERIOR_SAMPLES,
)
voo_posterior = mmar.stack_samples(voo_samples)

posterior_summary = pd.DataFrame(
    {
        "median": np.median(voo_posterior[0], axis=0),
        "5%": np.quantile(voo_posterior[0], 0.05, axis=0),
        "95%": np.quantile(voo_posterior[0], 0.95, axis=0),
    },
    index=mmar.PARAMETER_NAMES,
)
posterior_summary

In [ ]:
fig, axes = plt.subplots(
    1, 4, figsize=(12, 2.8), layout="constrained"
)
for index, (ax, name, label) in enumerate(
    zip(
        axes,
        mmar.PARAMETER_NAMES,
        mmar.PARAMETER_LABELS,
        strict=True,
    )
):
    ax.hist(
        voo_posterior[0, :, index],
        bins=35,
        density=True,
        color="#6F4AA8",
        alpha=0.8,
    )
    ax.set(title=name, xlabel=label, ylabel="Density")
fig.suptitle("VOO parameter posterior")

## 7. Posterior Predictive Check

Each posterior parameter draw generates a fresh 256-day return path. The predictive wealth bands show the range of paths implied by parameter uncertainty and new market shocks. The drawdown panel checks whether the observed path-level loss is plausible under those simulations.

In [ ]:
posterior_paths = mmar.posterior_resimulations(
    voo_posterior
)[0]

fig = mmar.plot_posterior_predictive(
    voo_returns,
    posterior_paths,
    voo_returns.index,
)

## 8. Gaussian Baseline

For a direct baseline, we fit a Gaussian distribution to the signed log-returns. If $x_t=\log(1+r_t)$ and $T=256$ is the number of trading days, the maximum-likelihood estimates are the sample mean and variance are simply:

$$
\widehat\mu_G=\frac{1}{T}\sum_{t=1}^{T}x_t,
\qquad
\widehat\sigma_G^2=\frac{1}{T}\sum_{t=1}^{T}
(x_t-\widehat\mu_G)^2.
$$

The left panel compares the observed absolute log-returns with the resulting folded Gaussian density. The right panel uses the same histogram but overlays densities from MMAR posterior predictive paths. This holds the observations fixed and changes only the fitted model.

In [ ]:
fig = mmar.plot_gaussian_mmar_comparison(
    voo_returns,
    posterior_paths,
    ticker="VOO",
    n_bins=40,
    hdi_probability=0.92,
)

## What We Learned

The workflow starts with a simulator, checks its prior implications, and validates the learned inverse map before applying it to market data. MMAR can assign more probability to extreme returns than the Gaussian baseline through two distinct mechanisms: the cascade clusters volatility, while the Student-t innovations thicken the marginal tails.

Agreement in the posterior predictive checks supports the model only for the summaries shown here. It does not prove that MMAR is the data-generating process, and forecasts outside the simulated parameter range should not be trusted without further validation.